In [1]:
import pandas as pd
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

import numpy as np
from scipy.spatial import ConvexHull

import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

In [2]:
# traceback "bonito" do IPython quebra (TokenError) ao formatar erros vindos
# de frames com fonte dinâmica (ex: lambdas de F.filter/F.transform do Spark)
# no Python 3.13 — Plain evita isso e mostra o erro real
%xmode Plain

Exception reporting mode: Plain


In [3]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .config("spark.python.worker.faulthandler.enabled", "true") # se algum worker crashar de novo, imprime o traceback nativo real
    .master("local[*]")
    .appName("features_analysis")
    .getOrCreate()
)

In [4]:
# caminhos pra pasta com dados
data_folder_path = Path().resolve().parent.parent / "data"
threat_dataset_path = str(data_folder_path / "threat_dataset")
features_dataset_path = str(data_folder_path / "features_dataset")

## Funções

In [5]:
# Funções de plot pra validação visual — uma por feature (+ base_defenders_figure,
# reaproveitada pelas Features 1-4). Não fazem parte do pipeline de cálculo
# (feature_engineering.ipynb), servem só pra conferir visualmente se o valor
# calculado bate com a geometria.


def base_defenders_figure(def_x, def_y, def_labels):
    """
    Cria a figura Plotly base reaproveitada pelas Features 1-4: só os
    defensores de linha (sem goleiro) plotados como markers rotulados pela
    posição.

    Parâmetros
    ----------
    def_x, def_y : np.ndarray
        Coordenadas dos defensores de linha (sem GK) do evento.
    def_labels : list[str]
        Sigla da posição de cada defensor, na mesma ordem de def_x/def_y.

    Retorna
    -------
    plotly.graph_objects.Figure
    """
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=def_x, y=def_y, mode='markers+text',
        text=def_labels, textposition='top center',
        marker=dict(size=12, color='blue'), name='Defensores (sem GK)'
    ))
    fig.update_yaxes(scaleanchor='x', scaleratio=1)
    fig.update_layout(height=550, width=700)
    return fig


def plot_surface_area(def_x, def_y, def_labels, surface_area):
    """
    Plota o casco convexo (Feature 1) sobre os defensores de linha (sem GK),
    preenchendo a área calculada.

    Parâmetros
    ----------
    def_x, def_y : np.ndarray
        Coordenadas dos defensores de linha (sem GK) do evento.
    def_labels : list[str]
        Sigla da posição de cada defensor, na mesma ordem de def_x/def_y.
    surface_area : float
        Valor de surface_area já calculado pra esse evento (exibido no título).
    """
    fig = base_defenders_figure(def_x, def_y, def_labels)

    pts = np.array(sorted(set(zip(def_x.tolist(), def_y.tolist()))))
    if len(pts) >= 3:
        hull = ConvexHull(pts)
        hull_pts = np.vstack([pts[hull.vertices], pts[hull.vertices[0]]])
        fig.add_trace(go.Scatter(
            x=hull_pts[:, 0], y=hull_pts[:, 1], mode='lines', fill='toself',
            fillcolor='rgba(0,0,255,0.1)', line=dict(color='blue', dash='dot'),
            name='Casco convexo'
        ))

    fig.update_layout(title=f"surface_area = {surface_area} m²")
    fig.show()


def plot_stretch_index(def_x, def_y, def_labels, stretch_index):
    """
    Plota o centroide dos defensores de linha (sem GK) e uma linha de cada
    defensor até ele (Feature 2).

    Parâmetros
    ----------
    def_x, def_y : np.ndarray
        Coordenadas dos defensores de linha (sem GK) do evento.
    def_labels : list[str]
        Sigla da posição de cada defensor, na mesma ordem de def_x/def_y.
    stretch_index : float
        Valor de stretch_index já calculado pra esse evento (exibido no título).
    """
    fig = base_defenders_figure(def_x, def_y, def_labels)

    cx, cy = def_x.mean(), def_y.mean()
    fig.add_trace(go.Scatter(x=[cx], y=[cy], mode='markers', marker=dict(size=12, symbol='x', color='purple'), name='Centroide'))
    for i, (x, y) in enumerate(zip(def_x, def_y)):
        fig.add_trace(go.Scatter(
            x=[cx, x], y=[cy, y], mode='lines', line=dict(color='purple', width=1),
            name='Distância até o centroide' if i == 0 else None, showlegend=(i == 0)
        ))

    fig.update_layout(title=f"stretch_index = {stretch_index} m (média das linhas roxas)")
    fig.show()


def plot_team_length(def_x, def_y, def_labels, team_length):
    """
    Plota linhas verticais tracejadas no defensor mais atrás e no mais à
    frente (eixo x) e um segmento horizontal ligando as duas (Feature 3).

    Parâmetros
    ----------
    def_x, def_y : np.ndarray
        Coordenadas dos defensores de linha (sem GK) do evento.
    def_labels : list[str]
        Sigla da posição de cada defensor, na mesma ordem de def_x/def_y.
    team_length : float
        Valor de team_length já calculado pra esse evento (exibido no título).
    """
    fig = base_defenders_figure(def_x, def_y, def_labels)

    i_min, i_max = int(def_x.argmin()), int(def_x.argmax())
    fig.add_vline(x=def_x[i_min], line_dash='dash', line_color='green', annotation_text='Defensor mais atrás')
    fig.add_vline(x=def_x[i_max], line_dash='dash', line_color='green', annotation_text='Defensor mais à frente')
    fig.add_trace(go.Scatter(
        x=[def_x[i_min], def_x[i_max]], y=[def_y.min() - 3, def_y.min() - 3],
        mode='lines+markers', line=dict(color='green', width=3), marker=dict(size=14, color='green'),
        name='team_length'
    ))

    fig.update_layout(title=f"team_length = {team_length} m")
    fig.show()


def plot_height_goal_player(def_x, def_y, def_labels, goal_x, height_goal_player):
    """
    Plota a linha do gol do time defendendo (x = goal_x) e um segmento até o
    defensor de linha mais próximo dela (Feature 4).

    Parâmetros
    ----------
    def_x, def_y : np.ndarray
        Coordenadas dos defensores de linha (sem GK) do evento.
    def_labels : list[str]
        Sigla da posição de cada defensor, na mesma ordem de def_x/def_y.
    goal_x : float
        Posição x do gol do time defendendo (stadiumLength / 2, já que o
        ataque está sempre normalizado pra direita).
    height_goal_player : float
        Valor de height_goal_player já calculado pra esse evento (exibido no título).
    """
    fig = base_defenders_figure(def_x, def_y, def_labels)

    i_closest = int(def_x.argmax())  # ataque normalizado pra direita -> gol em +goal_x, defensor mais próximo = maior x
    fig.add_vline(x=goal_x, line_dash='dash', line_color='black', annotation_text='Linha do gol')
    fig.add_trace(go.Scatter(
        x=[def_x[i_closest], goal_x], y=[def_y[i_closest], def_y[i_closest]],
        mode='lines+markers', line=dict(color='red', width=3), marker=dict(size=14, color='red'),
        name='height_goal_player'
    ))

    fig.update_layout(title=f"height_goal_player = {height_goal_player} m")
    fig.show()


def plot_numeric_superiority(all_attackers, all_defenders, ball, radius, value, radius_color='orange'):
    """
    Plota a bola, um círculo de raio `radius` ao redor dela e todos os
    atacantes/defensores (com goleiro) — usada pras Features 6 e 7
    (numeric_superiority_10m/20m).

    Parâmetros
    ----------
    all_attackers, all_defenders : list[Row]
        Jogadores completos (com GK) do time atacante/defendendo do evento,
        cada um com pelo menos os campos 'x' e 'y'.
    ball : Row
        Posição da bola do evento (campos 'x'/'y').
    radius : float
        Raio (m) ao redor da bola usado na contagem de superioridade numérica.
    value : float
        Valor de numeric_superiority_{radius}m já calculado pra esse evento
        (exibido no título).
    radius_color : str
        Cor da linha do círculo do raio (default 'orange').
    """
    fig = go.Figure()

    theta = np.linspace(0, 2 * np.pi, 100)
    fig.add_trace(go.Scatter(
        x=ball['x'] + radius * np.cos(theta), y=ball['y'] + radius * np.sin(theta),
        mode='lines', line=dict(color=radius_color, dash='dot'), name=f'Raio {radius}m'
    ))
    fig.add_trace(go.Scatter(
        x=[p['x'] for p in all_attackers], y=[p['y'] for p in all_attackers],
        mode='markers', marker=dict(size=10, color='red'), name='Atacantes (com GK)'
    ))
    fig.add_trace(go.Scatter(
        x=[p['x'] for p in all_defenders], y=[p['y'] for p in all_defenders],
        mode='markers', marker=dict(size=10, color='blue'), name='Defensores (com GK)'
    ))
    fig.add_trace(go.Scatter(x=[ball['x']], y=[ball['y']], mode='markers', marker=dict(size=14, color='black'), name='Bola'))

    fig.update_yaxes(scaleanchor='x', scaleratio=1)
    fig.update_layout(height=550, width=700, title=f"numeric_superiority_{radius}m = {value}")
    fig.show()

## Validação visual das features

In [6]:
# ============================================================
# Validação visual: escolhe 1 evento — cada célula abaixo chama a função de
# plot de UMA feature (definidas na seção "Funções"). feature_row vem do
# features_dataset já salvo por feature_engineering.ipynb — não recomputa
# nada aqui, só lê o CSV.
# ============================================================
sample_event_id = '248566037379f3842d4ba8870c387d82'  # troca aqui pra checar outro evento

sample_row = (
    spark.read.parquet(threat_dataset_path)
    .filter(F.col('eventId') == sample_event_id)
    .select('attackingPlayersNorm', 'defendingPlayersNorm', 'ballsNorm', 'stadiumLength')
    .first()
)

feature_row = (
    spark.read.csv(features_dataset_path, header=True, inferSchema=True)
    .filter(F.col('eventId') == sample_event_id)
    .first()
)

all_defenders = sample_row['defendingPlayersNorm']
all_attackers = sample_row['attackingPlayersNorm']
defenders_outfield = [p for p in all_defenders if p['position']['type'] != 'GK']
def_x = np.array([p['x'] for p in defenders_outfield])
def_y = np.array([p['y'] for p in defenders_outfield])
def_labels = [p['position']['type'] for p in defenders_outfield]
ball = sample_row['ballsNorm'][0]
goal_x = sample_row['stadiumLength'] / 2

In [7]:
# Feature 1 — surface_area
plot_surface_area(def_x, def_y, def_labels, feature_row['surface_area'])

In [8]:
# Feature 2 — stretch_index
plot_stretch_index(def_x, def_y, def_labels, feature_row['stretch_index'])

In [9]:
# Feature 3 — team_length
plot_team_length(def_x, def_y, def_labels, feature_row['team_length'])

In [10]:
# Feature 4 — height_goal_player
plot_height_goal_player(def_x, def_y, def_labels, goal_x, feature_row['height_goal_player'])

In [11]:
# Feature 6 — numeric_superiority_10m
plot_numeric_superiority(all_attackers, all_defenders, ball, radius=10, value=feature_row['numeric_superiority_10m'], radius_color='orange')

In [12]:
# Feature 7 — numeric_superiority_20m
plot_numeric_superiority(all_attackers, all_defenders, ball, radius=20, value=feature_row['numeric_superiority_20m'], radius_color='gray')